# Phase 1: YOLOX ROI extraction on the complete RSNA dataset (dual T4)
Kaggle, Internet ON, GPU ON. Inputs to attach: the RSNA competition + the dataset `dangnh0611/rsna-breast-cancer-detection-best-ckpts`.

All rows in `train.csv` are processed without creating train/validation/test splits. Extracted images are stored under `class_0` and `class_1`.


In [ ]:
import os
if not os.path.isdir("/kaggle/working/mammogram-cancer-classifier"):
    !git clone https://github.com/hassenmohamed-mezr/mammogram-cancer-classifier.git /kaggle/working/mammogram-cancer-classifier
%cd /kaggle/working/mammogram-cancer-classifier
!git pull
!pip -q install pydicom pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg
# if DICOM decoding fails later:  !pip -q install python-gdcm

### Add YOLOX (winner's breast detector)
1. Right sidebar -> **Add Input** -> search `rsna-breast-cancer-detection-best-ckpts` (by dangnh0611) -> add.
2. Run the next cell: installs YOLOX's small dependencies, clones the winner's repo (code only, ~80 MB, MIT licence) and finds the checkpoint.

In [ ]:
!pip -q install loguru thop pycocotools
WINNER = "/kaggle/working/rsna_winner"
if not os.path.isdir(WINNER):
    !git clone --depth 1 -b reproduce https://github.com/dangnh0611/kaggle_rsna_breast_cancer.git {WINNER}

import glob
hits = glob.glob("/kaggle/input/**/yolox_nano_416_roi_torch.pth", recursive=True)
print("checkpoint:", hits)
assert hits, "Add the best-ckpts dataset as an input (see step 1)"
CKPT = hits[0]

In [ ]:
!ls /kaggle/input
!ls /kaggle/input/competitions/rsna-breast-cancer-detection

In [ ]:
# ---- EDIT only if the ls above shows different names ----
COMP     = "/kaggle/input/competitions/rsna-breast-cancer-detection"
CSV      = f"{COMP}/train.csv"
IMG_ROOT = f"{COMP}/train_images"
IMG_TMPL = "{root}/{patient_id}/{image_id}.dcm"
OUT      = "/kaggle/working/all_images_manifest"


In [ ]:
# Build one manifest from every row in train.csv. No split and no sampling.
import pandas as pd
from pathlib import Path

Path(OUT).mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(CSV)
manifest["path"] = [
    IMG_TMPL.format(root=IMG_ROOT, patient_id=p, image_id=i)
    for p, i in zip(manifest["patient_id"], manifest["image_id"])
]
manifest.to_csv(f"{OUT}/roi_manifest.csv", index=False)


In [ ]:
print("Images in complete manifest:", len(manifest))
print("Class counts:")
print(manifest["cancer"].value_counts().sort_index().rename(index={0: "class_0", 1: "class_1"}))


In [ ]:
# YOLOX instance for the existing visual preview cell (GPU 0).
# The full extraction below launches one independent YOLOX process per GPU.
from src.dicom_roi import WinnerRoi, load_dicom, process
roi = WinnerRoi(WINNER, CKPT, device="cuda:0")
print("preview device:", roi.device)


In [ ]:
import numpy as np, matplotlib.pyplot as plt

rows = pd.concat([
    manifest[manifest.cancer == 1].sample(10, random_state=1),
    manifest[manifest.cancer == 0].sample(10, random_state=1)
])
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
    crop, info = process(load_dicom(r.path), roi)
    conf = "" if info["conf"] is None else f" {info['conf']:.2f}"
    ax.imshow(crop, cmap="gray"); ax.axis("off")
    ax.set_title(f"cancer={r.cancer} | {info['source']}{conf} | {crop.shape[1]}x{crop.shape[0]}", fontsize=9)
plt.tight_layout(); plt.show()


**Check the grid:** most titles should say `yolo`, the whole breast must be inside each crop, no inverted images.
If many say `otsu`/`full`, or boxes are wrong, stop and send me the grid.


## Full-dataset ROI extraction on both T4 GPUs
The complete manifest is divided only into two execution shards (one per GPU). This is workload distribution, not a train/validation/test data split. Each image is saved into its cancer-label directory.


In [ ]:
# Create a small worker script. Each worker loads its own YOLOX model on its assigned GPU.
from pathlib import Path

WORKER_SCRIPT = "/kaggle/working/run_roi_gpu_worker.py"
Path(WORKER_SCRIPT).write_text(r'''import argparse
import sys
import torch
sys.path.insert(0, "/kaggle/working/mammogram-cancer-classifier")
from src.dicom_roi import WinnerRoi
from src.run_roi import run

parser = argparse.ArgumentParser()
parser.add_argument("--gpu", type=int, required=True)
parser.add_argument("--manifest", required=True)
parser.add_argument("--out", required=True)
parser.add_argument("--winner", required=True)
parser.add_argument("--checkpoint", required=True)
args = parser.parse_args()

torch.cuda.set_device(args.gpu)
roi = WinnerRoi(args.winner, args.checkpoint, device=f"cuda:{args.gpu}")
run(args.manifest, args.out, roi=roi)
''' , encoding="utf-8")


In [ ]:
# Divide rows between the two independent GPU workers; every row is retained exactly once.
import os, sys, subprocess
from pathlib import Path

ROI_OUT = "/kaggle/working/roi_out"
WORKER_ROOT = Path("/kaggle/working/roi_workers")
WORKER_ROOT.mkdir(parents=True, exist_ok=True)

full_manifest = pd.read_csv(f"{OUT}/roi_manifest.csv")
manifest_0 = WORKER_ROOT / "gpu0_manifest.csv"
manifest_1 = WORKER_ROOT / "gpu1_manifest.csv"
full_manifest.iloc[0::2].to_csv(manifest_0, index=False)
full_manifest.iloc[1::2].to_csv(manifest_1, index=False)

processes = []
for gpu, shard in [(0, manifest_0), (1, manifest_1)]:
    worker_out = WORKER_ROOT / f"gpu{gpu}_output"
    command = [
        sys.executable, WORKER_SCRIPT,
        "--gpu", str(gpu),
        "--manifest", str(shard),
        "--out", str(worker_out),
        "--winner", WINNER,
        "--checkpoint", CKPT,
    ]
    processes.append(subprocess.Popen(command, cwd="/kaggle/working/mammogram-cancer-classifier"))

for process in processes:
    process.wait()


In [ ]:
# Merge both worker outputs and place every extracted crop under class_0 or class_1.
import shutil
from pathlib import Path

roi_root = Path(ROI_OUT)
class_0_dir = roi_root / "class_0"
class_1_dir = roi_root / "class_1"
class_0_dir.mkdir(parents=True, exist_ok=True)
class_1_dir.mkdir(parents=True, exist_ok=True)

result_parts = []
error_parts = []
for gpu in (0, 1):
    worker_out = Path(f"/kaggle/working/roi_workers/gpu{gpu}_output")
    part = pd.read_csv(worker_out / "roi_results.csv")
    for idx, row in part.iterrows():
        destination_dir = class_1_dir if int(row.cancer) == 1 else class_0_dir
        destination = destination_dir / f"{row.image_id}.png"
        shutil.move(str(row.roi_path), str(destination))
        part.at[idx, "roi_path"] = str(destination)
    result_parts.append(part)
    errors_path = worker_out / "roi_errors.csv"
    if errors_path.exists():
        error_parts.append(pd.read_csv(errors_path))

results = pd.concat(result_parts, ignore_index=True)
results.to_csv(roi_root / "roi_results.csv", index=False)
if error_parts:
    pd.concat(error_parts, ignore_index=True).to_csv(roi_root / "roi_errors.csv", index=False)

print("Extracted images:", len(results))
print("Class 0:", int((results.cancer == 0).sum()))
print("Class 1:", int((results.cancer == 1).sum()))
print("Errors recorded:", sum(len(x) for x in error_parts))
print("Output:", ROI_OUT)


**After the full run:** save `/kaggle/working/roi_out` as a Kaggle Dataset. It contains `class_0/`, `class_1/`, `roi_results.csv`, and (if any images failed) `roi_errors.csv`. The worker folders and manifests are under `/kaggle/working/roi_workers` and `/kaggle/working/all_images_manifest`.
